# Topic: LAG and LEAD

## Definition (30-second explanation)
* `LAG()` and `LEAD()` are window functions that allow you to access values from previous (LAG) or subsequent (LEAD) rows within a result set, relative to the current row.
* They allow for cross-row comparisons (like calculating period-over-period growth) without requiring computationally expensive self-joins.

## Why Interviewers Ask This
* To test your ability to perform time-series analysis and calculate standard business metrics (MoM, YoY growth).
* To see if you can write clean, performant SQL (using window functions instead of nested subqueries/joins).
* To verify you know how to handle edge cases like `NULL` values on the first/last rows and division-by-zero errors.

## Core Concepts
* **Offset:** The second parameter in the function (e.g., `LAG(revenue, 1)`) determines how many rows to look back/forward. The default is 1.
* **Default Value:** The third parameter (e.g., `LAG(revenue, 1, 0)`) defines what to return if the target row doesn't exist (e.g., the very first row), preventing unexpected `NULL`s.
* **PARTITION BY:** Critical for resetting the window per group (e.g., looking at previous revenue *for the same product*, not just the absolute previous row).

## When to Use
* **LAG:** Calculating day-over-day or month-over-month changes, or finding the time elapsed between consecutive user events (sessionization).
* **LEAD:** Looking forward to predict next states, checking if a user upgraded in their next billing cycle, or identifying trend reversals.

## Advantages
* Dramatically simplifies SQL logic compared to self-joins.
* Highly performant since the database engine only needs to scan the data partition once.
* CTEs combined with LAG/LEAD make complex trend logic highly readable.

## Limitations
* Requires a strict, deterministic `ORDER BY` inside the `OVER()` clause to ensure adjacent rows are chronologically correct.
* The output is sensitive to missing data (e.g., if a month is missing from the table, `LAG(..., 1)` will grab the month before that, skipping a month).

## Common Comparisons
* **LAG vs LEAD:** `LAG` looks backwards (historical comparison), `LEAD` looks forwards (future outcome tracking). 
* **LAG vs Self-Join:** LAG is linear and clean; Self-Joins require joining a table on `t1.date = t2.date + 1`, which is slow and prone to duplication if relationships aren't 1:1.

## Common Interview Traps
* **Missing PARTITION BY:** If omitted, `LAG` might compare the January revenue of a 'Phone' to the December revenue of a 'Laptop' just because they are adjacent in the table.
* **Division by Zero:** When calculating growth percentage `(current - prev) / prev`, if `prev` is 0, the query crashes. Always use `NULLIF(prev, 0)`.
* **Cluttered SELECT clauses:** Writing the full `LAG(...)` window function multiple times in the same `SELECT` (once for absolute change, once for percentage). Use a CTE to define it once.

## Python / SQL Syntax
```sql
-- Standard syntax with offset (1) and default value (0)
LAG(column_name, 1, 0) OVER (
    PARTITION BY group_col 
    ORDER BY sort_col
)
```

## Important Formula
**Percentage Change:** (current_value - previous_value) / NULLIF(previous_value, 0)

## 45-Second Interview Answer
"LAG and LEAD are window functions used to access data from adjacent rows without needing self-joins. LAG looks backward, which is perfect for period-over-period growth or time-between-events, while LEAD looks forward. In interviews, the key to using them correctly is ensuring you have a strict ORDER BY for chronological alignment, a proper PARTITION BY so you don't accidentally compare different categories, and handling edge cases—like providing a default value for the first row to prevent NULLs, and using NULLIF to prevent division-by-zero when calculating growth rates."

## Example Questions:
**Mock Schema:**
```sql
-- DDL
CREATE TABLE monthly_sales (
    product VARCHAR(50),
    sale_month DATE,
    revenue DECIMAL(10, 2)
);

-- Mock Data Insertion
INSERT INTO monthly_sales (product, sale_month, revenue) VALUES
-- ==========================================
-- PRODUCT: Laptop
-- ==========================================
('Laptop', '2025-01-01', 45000.00),
('Laptop', '2025-02-01', 52000.00), -- Grew
('Laptop', '2025-03-01', 48000.00), -- Dropped (Peak reversal in Feb)
('Laptop', '2025-04-01', 61000.00), -- Grew (Valley reversal in Mar)
('Laptop', '2025-05-01', 65000.00), -- Grew (Consecutive = 2)
('Laptop', '2025-06-01', 70000.00), -- Grew (Consecutive = 3)
('Laptop', '2025-07-01', 40000.00), -- Dropped dramatically (> 10% drop)
('Laptop', '2026-01-01', 55000.00), -- For YoY comparison against 2025-01
('Laptop', '2026-02-01', 60000.00), -- For YoY comparison against 2025-02

-- ==========================================
-- PRODUCT: Phone
-- ==========================================
('Phone', '2025-01-01', 38000.00),
('Phone', '2025-02-01', 41000.00),  -- Grew (Consecutive = 1)
('Phone', '2025-03-01', 45000.00),  -- Grew (Consecutive = 2)
('Phone', '2025-04-01', 55000.00),  -- Grew (Consecutive = 3)
('Phone', '2025-05-01', 60000.00),  -- Grew (Consecutive = 4 - longest streak!)
('Phone', '2025-06-01', 30000.00),  -- Dropped dramatically (> 10% drop)
('Phone', '2025-07-01', 35000.00),  -- Grew (Valley reversal in June)
('Phone', '2026-01-01', 48000.00),  -- For YoY comparison against 2025-01
('Phone', '2026-02-01', 50000.00);  -- For YoY comparison against 2025-02
```

### Q1. Calculate year-over-year revenue change for each product.

**Answer1:**
```sql
WITH lagged_data AS (
    SELECT 
        product, 
        sale_month, 
        revenue,
        LAG(revenue, 12) OVER(PARTITION BY product ORDER BY sale_month) AS prev_yr_revenue
    FROM monthly_sales
)
SELECT 
    product, 
    sale_month, 
    revenue, 
    prev_yr_revenue,
    100.0 * (revenue - prev_yr_revenue) / NULLIF(prev_yr_revenue, 0) AS yoy_growth
FROM lagged_data
WHERE prev_yr_revenue IS NOT NULL;
```

**Answer2: (Takes care of scenario if months are missing between)**
```sql
with lagged_revenue as (
	select m1.product, m1.sale_month, m1.revenue, m2.revenue as prev_revenue
  	from monthly_sales m1 join monthly_sales m2
  	on (m1.product = m2.product)
  	and m2.sale_month = m1.sale_month - interval 12 month
)
select product, sale_month, revenue, prev_revenue,
	100.0 * (revenue - prev_revenue) / nullif(prev_revenue, 0) as YoY_growth
from lagged_revenue;
```
* **Common Mistakes:** Using an offset of 12 on daily/monthly data without explicitly aggregating it to the yearly level first, which breaks if a month/day is missing.
* **Likely Follow-up:** "How would you write this if you wanted Month-over-Month growth, but some products have missing months in the dataset?"

### Q2. Find all months where revenue dropped more than 10% from the previous month.

**Answer:**
```sql
WITH lagged_revenue AS (
    SELECT 
        product, 
        sale_month, 
        revenue,
        LAG(revenue) OVER(PARTITION BY product ORDER BY sale_month) AS prev_revenue
    FROM monthly_sales
)
SELECT 
    product, 
    sale_month, 
    revenue, 
    prev_revenue
FROM lagged_revenue
WHERE revenue < prev_revenue * 0.90;
```
* **Common Mistakes:** Trying to put the `LAG` function directly inside the `WHERE` clause, which SQL does not allow.
* **Likely Follow-up:** "Why use a CTE instead of writing the LAG function twice in the SELECT and WHERE clauses?"

### Q3. Calculate a 3-month moving average using LAG.

**Answer:**
```sql
SELECT 
    product, 
    sale_month, 
    revenue,
    ROUND(AVG(revenue) OVER(
        PARTITION BY product 
        ORDER BY sale_month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS three_month_moving_avg
FROM monthly_sales;
```
* **Common Mistakes:** Hardcoding `LAG(..., 1)` and `LAG(..., 2)` which becomes unscalable if the requirement changes to a 12-month moving average.
* **Likely Follow-up:** "What happens to the moving average for the very first month of a product's lifecycle using the `ROWS BETWEEN` method?"

### Q4. Detect revenue trend reversals (went up then down, or down then up).
**Answer:**
```sql
WITH lagged_lead_revenue AS (
    SELECT 
        product, 
        sale_month, 
        revenue,
        LAG(revenue) OVER(PARTITION BY product ORDER BY sale_month) AS prev_rev,
        LEAD(revenue) OVER(PARTITION BY product ORDER BY sale_month) AS next_rev
    FROM monthly_sales
)
SELECT 
    product, 
    sale_month, 
    revenue,
    prev_rev,
    next_rev
FROM lagged_lead_revenue
WHERE (revenue < prev_rev AND revenue < next_rev) -- Valley
   OR (revenue > prev_rev AND revenue > next_rev); -- Peak
```
* **Common Mistakes:** Overcomplicating it with multiple nested CTEs instead of grabbing both the prior and next values in a single CTE pass.
* **Likely Follow-up:** "How would you handle a scenario where the revenue stayed exactly flat for a month before dropping?"

### Q5. Find the product with the most consecutive months of growth.
**Answer:** This is a classic "gaps and islands" problem. First, I'd use `LAG` to flag if a month grew compared to the previous (1 for growth, 0 for no growth). Then, I'd use a cumulative sum (`SUM() OVER()`) of the *non-growth* flags to group consecutive growth months into unique "islands". Finally, I'd count the size of each island, rank them, and select the maximum.

```sql
WITH lagged_data AS (
    SELECT 
        product, 
        sale_month, 
        revenue,
        LAG(revenue) OVER(PARTITION BY product ORDER BY sale_month) AS prev_rev
    FROM monthly_sales
),
flags AS (
    SELECT 
        product,
        sale_month,
        CASE WHEN revenue > prev_rev THEN 1 ELSE 0 END AS is_growth,
        -- We sum this flag to create the Island ID
        CASE WHEN revenue <= prev_rev OR prev_rev IS NULL THEN 1 ELSE 0 END AS is_drop
    FROM lagged_data
),
islands AS (
    SELECT 
        product,
        is_growth,
        -- The running total of drops creates a unique ID for each growth streak
        SUM(is_drop) OVER(PARTITION BY product ORDER BY sale_month) AS island_id
    FROM flags
)
SELECT 
    product, 
    COUNT(*) AS consecutive_months
FROM islands
WHERE is_growth = 1
GROUP BY product, island_id
ORDER BY consecutive_months DESC
LIMIT 1;
```
* **Common Mistakes:** Thinking this can be solved with a single `LAG` statement without recognizing it requires iterative grouping logic (gaps and islands).
* **Likely Follow-up:** "Can you write the pseudo-code for the 'islands' grouping step?"

## Practice Questions:

### Q1:
**Scenario:**
You are a Data Scientist analyzing customer retention. The marketing team wants to know the number of days that elapsed between a user's current purchase and their immediately preceding purchase. If it is the user's very first purchase, the "days since last purchase" should be 0.

**Mock Schema:**
```sql
-- DDL
CREATE TABLE purchases (
    purchase_id INT AUTO_INCREMENT PRIMARY KEY,
    user_id INT,
    purchase_date DATE,
    amount DECIMAL(10, 2)
);

-- Mock Data
INSERT INTO purchases (user_id, purchase_date, amount) VALUES
(101, '2026-08-01', 50.00),
(101, '2026-08-05', 75.50),  -- 4 days since last
(101, '2026-08-15', 20.00),  -- 10 days since last
(102, '2026-08-02', 100.00), -- 0 days since last
(102, '2026-08-03', 45.00),  -- 1 day since last
(103, '2026-08-10', 60.00);  -- 0 days since last
```

* **Answer:** 
```sql
WITH prev_purchases AS (
    SELECT 
        user_id, 
        purchase_date,
        LAG(purchase_date) OVER(PARTITION BY user_id ORDER BY purchase_date) as prev_purchase
    FROM purchases
)
SELECT 
    user_id, 
    purchase_date,
    COALESCE(DATEDIFF(purchase_date, prev_purchase), 0) AS days_since_last_purchase
FROM prev_purchases;
```
* **Common Mistakes:** 
    * Using a `GROUP BY` clause, which collapses the transaction-level data and destroys the intermediate purchase history.
    * Forgetting to handle the `NULL` value on the user's very first row, which results in a `NULL` calculation instead of the requested `0`.
* **Interview Tip:** Whenever calculating "time since last [event]" (often called sessionization), `LAG` is your best friend. Wrap it in a CTE, and simply run a date difference function on the outer query. Never use `GROUP BY` unless you are actively trying to reduce the number of rows (e.g., finding only the maximum purchase).

### Q2:
**Scenario:**
You are analyzing monthly sales for different products. The Finance team wants a report showing the product_id, sale_month, revenue, and the Month-over-Month (MoM) growth_percentage.

**Mock Schema:**
```sql
CREATE TABLE monthly_sales (
    product_id INT,
    sale_month DATE,
    revenue DECIMAL(10, 2)
);

INSERT INTO monthly_sales (product_id, sale_month, revenue) VALUES
(1, '2026-01-01', 1000.00),
(1, '2026-02-01', 1500.00), -- Grew 50%
(1, '2026-03-01', 1200.00), -- Dropped -20%
(2, '2026-01-01', 500.00),
(2, '2026-02-01', 0.00),    -- Dropped to 0
(2, '2026-03-01', 200.00);  -- Revenue comes back, must avoid division by zero!
```

**Question:**
Write the MySQL query to generate this report.

* **Answer:** 
```sql
WITH prev_month_data AS (
    SELECT 
        product_id, 
        sale_month, 
        revenue,
        LAG(revenue) OVER(PARTITION BY product_id ORDER BY sale_month) AS prev_revenue
    FROM monthly_sales
)
SELECT 
    product_id, 
    sale_month, 
    revenue, 
    prev_revenue,
    100.0 * (revenue - prev_revenue) / NULLIF(prev_revenue, 0) AS mom_growth
FROM prev_month_data;
```
* **Common Mistakes:** 
    * Forgetting the `NULLIF(prev_revenue, 0)` in the denominator, which causes the query to fail violently in production as soon as a $0 value is encountered.
    * Trying to write the math using `LAG()` directly in the `SELECT` clause without a CTE, which makes the query incredibly messy and hard to read.
* **Interview Tip:** Whenever calculating growth rates, ratios, or percentages in SQL interviews, loudly state: *"I'm going to wrap the denominator in a NULLIF to protect against division by zero."* Interviewers will instantly mentally check off a "seniority/experience" box.

### Q3:
**Scenario:**
You are analyzing daily active users (DAU). The product team wants to identify "Peaks." A peak is defined as a day where the active_users count was strictly greater than the day before it, AND strictly greater than the day after it.

**Mock Schema:**
```sql
CREATE TABLE daily_metrics (
    metric_date DATE,
    active_users INT
);

INSERT INTO daily_metrics (metric_date, active_users) VALUES
('2026-08-01', 100),
('2026-08-02', 120),
('2026-08-03', 150), -- Peak! (Higher than Aug 2 and Aug 4)
('2026-08-04', 130),
('2026-08-05', 110),
('2026-08-06', 140), -- Peak! (Higher than Aug 5 and Aug 7)
('2026-08-07', 125);
```

* **Answer:** 
```sql
WITH prev_next AS (
    SELECT 
        metric_date, 
        active_users,
        LAG(active_users) OVER(ORDER BY metric_date) AS prev_active_users,
        LEAD(active_users) OVER(ORDER BY metric_date) AS next_active_users
    FROM daily_metrics
)
SELECT 
    metric_date, 
    active_users
FROM prev_next
WHERE active_users > prev_active_users 
  AND active_users > next_active_users;
```
* **Common Mistakes:** 
    * Trying to join the table to itself twice (once for the previous day, once for the next day). While this works, it is computationally expensive and shows a lack of window function knowledge.
    * Forgetting the `ORDER BY metric_date` inside the `OVER()` clause, which makes the previous/next row essentially random based on how the database engine stored the data.
* **Interview Tip:** If the interviewer asks "What happens to the first and last dates in the table?", you can confidently explain that SQL evaluates comparisons with `NULL` as `UNKNOWN`, meaning the first and last dates will be naturally filtered out—which is the mathematically correct behavior for a local maximum!